# Graph Inspection — ad hoc diagnostic notebook

Not a pipeline stage (no number prefix, doesn't feed `05`/`06`/`07`) —
just a place to browse `graph_inspect.py`'s tables and topology diagrams
against real saved graphs. Safe to run at any point after `04`/`04b`
have produced `.pt` files; needs no GPU.

Two independent things this answers:
- **What's actually inside a given point's graph** — every node, every
  edge, every attribute, decoded (`type_idx=173` → `"yes"`), as pandas
  tables and CSVs.
- **What that graph looks like structurally** — a diagram, node type by
  node type, edge relation by edge relation. Not a spatial rendering
  (see `svg_visualize.py`/`tvg_visualize.py` for that) — a topology view
  that needs nothing but the saved tensors.

In [ ]:
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

### TEMP: upload `graph_inspect.py` until it's pushed to GitHub

Skip this cell once `src/graph_inspect.py` is committed — the check
below makes it a no-op at that point rather than something you have to
remember to delete.

In [ ]:
from pathlib import Path

target = Path(REPO_DIR) / "src" / "graph_inspect.py"
if target.exists():
    print(f"{target} already present (committed) — skipping upload.")
else:
    from google.colab import files
    import shutil
    print("Upload graph_inspect.py:")
    uploaded = files.upload()
    for fname in uploaded:
        shutil.move(fname, str(target))
    print("Installed:", list(uploaded.keys()))

In [ ]:
# matplotlib is already a project dependency; networkx usually rides in
# via osmnx (already required by 04/tvg_builder) but installed explicitly
# here too so this notebook doesn't depend on that being true.
!pip install -q matplotlib networkx pyyaml pandas

In [ ]:
import yaml
from pathlib import Path

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)

print("cities configured:", paths_cfg["cities"])

### Pick ONE dataset to browse: single-city or pooled

Run exactly one of the next two cells depending on what you want to
look at. Both leave a `dataset` and a `vocabs` dict behind, which is all
`graph_inspect` needs from here on — the rest of the notebook doesn't
care which one you picked.

In [ ]:
# ── Option A: single city (matches 07's dataset_index.parquet) ──────
import pandas as pd
import graph_datasets as ds
import graph_inspect as gi

CITY = "bogor"  # or "warsaw"
city_cfg = paths_cfg["per_city"][CITY]
processed_dir = Path(city_cfg["processed_dir"])
cache_dir = Path(city_cfg["base_dir"]) / "interim" / "osm_cache"

index_df = pd.read_parquet(processed_dir / "dataset_index.parquet")
dataset = ds.DualGraphDataset(index_df, processed_dir / "svg_graphs", processed_dir / "tvg_graphs")

vocabs = gi.load_vocabs(
    building_vocab_path=cache_dir / "building_type_vocab.json",
    highway_vocab_path=cache_dir / "highway_vocab.json",
)
print(f"{CITY}: {len(dataset)} points loaded.")

In [ ]:
# ── Option B: pooled Bogor+Warsaw (matches 07e/07f's pooled index) ──
import pandas as pd
import graph_datasets as ds
import graph_inspect as gi
import dataset_audit as da

_bogor_base = Path(paths_cfg["per_city"]["bogor"]["base_dir"])
combined_processed_dir = _bogor_base.parent / "combined" / "processed"

index_df = pd.read_parquet(combined_processed_dir / "dataset_index.parquet")
dataset = ds.PooledDualGraphDataset(index_df)

# Reads the SAME unified vocab from both cities' caches and asserts they
# agree — see dataset_audit.load_unified_vocab_sizes's docstring for why
# that agreement check matters once you're pooling two cities.
city_cache_dirs = {
    city: Path(paths_cfg["per_city"][city]["base_dir"]) / "interim" / "osm_cache"
    for city in paths_cfg["cities"]
}
unified = da.load_unified_vocab_sizes(city_cache_dirs)
vocabs = gi.load_vocabs(
    building_vocab_path=list(city_cache_dirs.values())[0] / "building_type_vocab.json",
    highway_vocab_path=list(city_cache_dirs.values())[0] / "highway_vocab.json",
)
print(f"pooled: {len(dataset)} points loaded across {sorted(index_df['city'].unique())}.")

## Browse — find a point worth looking at

In [ ]:
# A few starting points — mix and match filters as needed.
gi.browse_points(dataset, n=10)                              # first 10, whatever order the index is in
# gi.browse_points(dataset, city="warsaw", label=1)           # every Warsaw positive
# gi.browse_points(dataset, n=5, random_state=0)               # reproducible random sample
# gi.browse_points(dataset, uid_contains="positive_4")         # substring match on uid/point_id

## Inspect one point — full tabular dump

In [ ]:
UID = gi.browse_points(dataset, n=1, random_state=0).iloc[0]["uid" if "uid" in dataset.index_df.columns else "point_id"]
print(f"Inspecting: {UID}")

dump = gi.inspect_point(dataset, UID, vocabs=vocabs)
# add export_dir="inspect_out/" + UID  to also write every node/edge type
# out as CSVs, e.g. for opening a point next to a paper figure.

### Pull one node or edge type's full table directly, if `inspect_point`'s
combined printout is more than you need for this pass.

In [ ]:
svg_data, tvg_data, label, uid = dataset[dataset.index_df.index[
    dataset.index_df["uid" if "uid" in dataset.index_df.columns else "point_id"] == UID][0]]

gi.node_table(svg_data, "signage", schema="svg", vocabs=vocabs)
# gi.edge_table(tvg_data, ("intersection", "connects", "intersection"), schema="tvg", vocabs=vocabs)

## Visualize one point — topology diagrams

Just call it as the last line of a cell — Colab's inline matplotlib
backend renders the returned figure automatically, no `plt.show()` or
`display()` needed.

In [ ]:
gi.plot_point(dataset, UID, vocabs=vocabs)
# add save_dir="figs/" to also write the PNG to disk

In [ ]:
# Single branch, bigger, if you want to zoom in on just one side:
gi.plot_graph(svg_data, schema="svg", vocabs=vocabs, title=f"SVG — {UID}", figsize=(10, 9))

## Compare a few points side by side

The cheapest way to eyeball "do positive and negative points actually
look structurally different" before spending time on architecture
changes — loop `plot_point` over a small `browse_points` sample.

In [ ]:
sample = gi.browse_points(dataset, n=3, random_state=1)
id_col = "uid" if "uid" in dataset.index_df.columns else "point_id"
for _, row in sample.iterrows():
    gi.plot_point(dataset, row[id_col], vocabs=vocabs)

---
### Known gap: no "show me what the model got wrong" mode yet

`browse_points` filters by label/city/uid — it can't yet filter by
prediction correctness, because `train.py`'s `_run_epoch_eval` doesn't
currently carry point ids alongside `y_true`/`y_prob`. Cross-referencing
false positives/negatives against these diagrams is the natural next
diagnostic step once that's wired up.